# `validate_session` — per-step checks for a `banish_multiplier` session

One section per pipeline step, in the order `run_session.py` runs them. Each section prints what the
step produced next to the value it is **expected** to produce, and marks `PASS` / `**FAIL**` /
`note`. The reference numbers are JPAS_0168's validated run (see `CLAUDE.md`); on a **new animal**
they will differ — that is the point. A `FAIL` there is not a bug, it is the signal that the step is
**per-animal tuned** and needs a human to look at the picture before anything downstream is trusted.

**Which steps can silently go wrong on a new mouse** (the `QC` gates in `run_session.py --list`):
`pupil`, `eye`, `groom`, `lick`, `mouth`. Every one of those has a clip or a contact sheet in this
notebook — *look at it*, don't just read the number. Every pupil-fit bug in this project was caught
by eyeballing an overlay, never by a statistic.

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SESSION = Path('.').resolve()          # <-- run this notebook from inside the session directory
PIPE    = SESSION.parent / 'session_pipeline'
for p in (PIPE/'common', PIPE/'common'/'detectors', PIPE/'task_banish_multiplier'):
    sys.path.insert(0, str(p))

sess = json.load(open(SESSION/'session.json'))
OD   = SESSION/'opticflow'
FPS  = sess['fps']
print(f"session {sess['mouse_id']}   task={sess.get('task_type')}   fps={FPS}")

_n_pass = _n_fail = 0
def check(name, actual, expected=None, tol=None, unit=''):
    '''Print actual vs expected. `tol` is an ABSOLUTE tolerance; give expected=None for a value
    that is informational only (no reference to compare against on a new animal).'''
    global _n_pass, _n_fail
    if expected is None:
        print(f'  note   {name:44s} = {actual}{unit}'); return
    ok = abs(actual - expected) <= (tol if tol is not None else 0)
    _n_pass, _n_fail = _n_pass + ok, _n_fail + (not ok)
    print(f"  {'PASS  ' if ok else '**FAIL' + '**'} {name:44s} = {actual}{unit}   "
          f"(expected {expected}{unit}{'' if tol is None else f' +-{tol}'})")

def summary():
    print(f'\n=== {_n_pass} passed, {_n_fail} failed vs the JPAS_0168 reference ===')

## 1. `session` — session.json + camera check

The contract every later step reads. If the camera MOVED mid-session the framing drifts and every ROI-based signal after the onset frame is measuring a different patch of face — that session must not be analyzed (JPAS_0168_cameraMoved is the kept counter-example).

In [ ]:
cc = json.load(open(OD/'camera_check.json')) if (OD/'camera_check.json').exists() else {}
check('fps', FPS, 30.0, 0.01)
check('n_frames', int(sess['n_frames']), 77138, 0)
check('frame width', int(sess['width']), 1280, 0)
check('world size', int(sess['world_width']), 2400, 0)
check('camera_moved (0 = stable)', int(bool(sess.get('camera_moved'))), 0, 0)
# view_scale is a property of the WORLD and differs per session; viewport.py refuses to
# default it, because a silent default would let this session inherit another world's zoom
# and corrupt every visibility-gated result (icon attention, the performance baseline).
import viewport as vp
check('view_scale (per-WORLD, must be set)', vp.view_scale(sess), 0.35, 0.001)
print(f'  viewport = {vp.viewport(sess)[0]:.0f} x {vp.viewport(sess)[1]:.0f} world units'
      f"   [{sess.get('view_scale_source','')}]")
print('  ROIs:', ', '.join(sess['rois']))
assert not sess.get('camera_moved'), 'CAMERA MOVED - do not analyze this session'

## 2. `flow` — per-ROI optic flow

Farneback per ROI, 8 metrics each. The check is that every ROI produced a full-length, non-degenerate magnitude trace: an all-zero or truncated ROI means the crop/pad went wrong.

In [ ]:
rois = ['whisker_left','whisker_right','nose','paw','mouth','left_eye','left_fovea']
n = int(sess['n_frames'])
for r in rois:
    m = np.load(OD/f'opticflow_{r}_mag.npy')
    ok = len(m) == n and np.isfinite(m).all() and m.std() > 0
    print(f"  {'PASS  ' if ok else '**FAIL**'} {r:14s} len={len(m):6d}  mean={m.mean():6.3f}  "
          f"p99={np.percentile(m,99):6.3f}")

## 3. `pupil` — the fit  ⚠️ HUMAN-QC GATE

The glint-anchored dark-iris fit. **Per-animal tuned.** A fit can be stable and confidently wrong (it can lock onto the socket shadow), so the number below is necessary but NOT sufficient — open `debug/pupil_fit_check.png` and look at the circles on the RAW crops.

In [ ]:
tr = np.load(OD/'pupil_track.npz', allow_pickle=True)
r  = tr['radius'].astype(float)
valid = np.isfinite(r)
check('valid fit fraction', round(float(valid.mean())*100, 1), 100.0, 1.0, '%')
check('radius median (px)', round(float(np.nanmedian(r)), 1), 9.9, 0.6, 'px')
check('glint present', round(float(tr['glint_present'].mean())*100, 1), 100.0, 1.0, '%')
check('radius p5', round(float(np.nanpercentile(r,5)),1), 9.2, 0.8, 'px')
check('radius p95', round(float(np.nanpercentile(r,95)),1), 11.9, 1.2, 'px')
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(r[::20], lw=.4); ax[0].set_title('radius over the session (every 20th frame)'); ax[0].set_ylabel('px')
ax[1].hist(r[valid], 60); ax[1].set_title('radius distribution — a hard second mode = a bistable fit')
plt.tight_layout(); plt.show()
print('\n  ⚠️ NOW LOOK AT debug/pupil_fit_check.png — a statistic cannot tell you the circle is on the iris.')

## 4. `eye` — blink / squint / position  ⚠️ HUMAN-QC GATE

Blink comes from `eye_mean` (the whole eye box), NOT fovea brightness — the small fovea box barely moves during a blink. Each closure is then split into a full BLINK vs a milder EYE SQUINT by the pupil-fit shape. The `median+12` threshold is **per-animal**.

In [ ]:
ev = np.load(OD/'eye_events.npz', allow_pickle=True)
blink, squint = ev['blink'], ev['squint']
mins = int(sess['n_frames'])/FPS/60
check('blink frames', int(blink.sum()), 131, 12)
check('squint frames', int(squint.sum()), 31, 10)
check('closure frames (blink+squint)', int(blink.sum()+squint.sum()), 162, 15)
check('blink events', len(ev['blink_spans']), 20, 5)
check('blink rate (/min)', round(len(ev['blink_spans'])/mins, 2), 0.47, 0.25, '/min')
print('  note   this mouse blinks RARELY — a rate near 0 is expected here, not a broken detector')

## 5. `whisker` — sweep + cycle

3–13 Hz band-pass → principal axis → Hilbert. The cycle must land in the whisking band and stay below the 15 Hz Nyquist; a cycle pinned near Nyquist means the band-pass or fps is wrong.

In [ ]:
wz = np.load(OD/'whisker.npz', allow_pickle=True)
sweep, cyc = wz['sweep'], wz['cycle_whisker_right']
check('cycle median (Hz)', round(float(np.nanmedian(cyc)), 2), 7.0, 1.5, 'Hz')
check('cycle below Nyquist 15 Hz', int(np.nanmedian(cyc) < 15), 1, 0)
check('whisking fraction', round(float(wz['whisking'].mean())*100, 1), None, unit='%')
mag = np.load(OD/'opticflow_whisker_right_mag.npy')
check('corr(sweep, whisker |flow|)', round(float(np.corrcoef(sweep, mag)[0,1]), 2), 0.80, 0.20)

## 6. `groom` — grooming  ⚠️ HUMAN-QC GATE

paw & whisker |flow| both > p75, **gated on joystick stillness**. The gate is what removes the locomotion confound: without it, joystick speed during 'grooming' comes out HIGHER than at rest, which is the giveaway that the detector is finding movement, not grooming.

In [ ]:
gm = np.load(OD/'groom_mask_clean.npy')
spans = np.diff(np.concatenate([[0], gm.astype(int), [0]]))
n_bouts = int((spans == 1).sum())
check('grooming fraction', round(float(gm.mean())*100, 1), 1.7, 0.8, '%')
check('grooming bouts', n_bouts, 73, 25)
# the confound test: joystick speed must be LOWER during grooming than at rest
import fps as fpsmod
log = json.load(open(SESSION/'log.json'))
fm  = fpsmod.frame_to_ms(log)
J   = np.array(log['joystick_t[ms]/x/y'], float)
spd = np.interp(fm, J[:,0], np.hypot(np.gradient(J[:,1]), np.gradient(J[:,2])))
g, rest = spd[gm[:len(spd)]].mean(), spd[~gm[:len(spd)]].mean()
print(f"  {'PASS  ' if g < rest else '**FAIL**'} joystick speed during grooming {g:.3f} < at rest {rest:.3f}"
      '   <-- if this FLIPS, the stillness gate is broken and you are detecting locomotion')

## 7. `lick` — licking  ⚠️ HUMAN-QC GATE

Mouth `fy` 5–12 Hz rhythm envelope. The strong self-check: the **within-bout frequency** is measured independently of the band used to find the bouts, so if they disagree the band or the `fy` sign is wrong for this animal.

In [ ]:
lz = np.load(OD/'licking.npz', allow_pickle=True)
bout = lz['lick_bout'].astype(bool)
check('licking fraction', round(float(bout.mean())*100, 1), 20.6, 5.0, '%')
check('lick bouts', len(lz['bout_spans']), 956, 250)
# Within-bout frequency, recomputed HERE from the detected lick peaks. This is the strong
# self-check: the spacing of the peaks is not what the 5-12 Hz band was tuned on, so if it
# disagrees with the known lick rate the band or the `fy` sign is wrong for this animal.
peaks = np.asarray(lz['lick_frames'])
hz = [FPS / np.median(np.diff(pk)) for s, e in np.atleast_2d(lz['bout_spans'])
      for pk in [peaks[(peaks >= s) & (peaks <= e)]] if len(pk) > 2]
check('within-bout frequency (recomputed)', round(float(np.median(hz)), 1), 7.7, 1.5, 'Hz')

## 8. `mouth` — single mouth axis  ⚠️ HUMAN-QC GATE

0 CLOSED / 1 LICKING / 2 GROOMING with **GROOMING PRIORITISED**. A raised paw sweeps the mouth ROI and false-fires the flow lick detector, so overlapping frames are given to grooming. The count of REASSIGNED frames is the thing to eyeball in the clip below — the paw must be up on all of them.

In [ ]:
ms = np.load(OD/'mouth_state.npz', allow_pickle=True)['mouth_state']
for v, lab, exp in [(0,'CLOSED',79.0), (1,'LICKING',19.3), (2,'GROOMING',1.7)]:
    check(f'{lab} fraction', round(float((ms==v).mean())*100, 1), exp, 5.0, '%')
lick_flow = np.load(OD/'licking.npz', allow_pickle=True)['lick_bout'].astype(bool)[:len(ms)]
check('frames reassigned lick -> groom', int((lick_flow & (ms==2)).sum()), 1046, 400)
assert set(np.unique(ms)) <= {0,1,2}, 'mouth_state must be a SINGLE axis with no contradictory pairs'

## 9. `saccade` — APPROXIMATE P-CR saccades

Flagged approximate everywhere it is used: the pupil centre is itself approximate for this mouse, so the count is likely inflated. Blink and squint frames are excluded by construction.

In [ ]:
sz = np.load(OD/'saccades.npz', allow_pickle=True)
sac = sz['saccade']
mins = int(sess['n_frames'])/FPS/60
check('saccades', len(sac), 227, 80)
check('rate (/min)', round(len(sac)/mins, 1), 5.3, 2.0, '/min')
ev = np.load(OD/'eye_events.npz', allow_pickle=True)
bad = int((ev['blink'][sac] | ev['squint'][sac]).sum())
print(f"  {'PASS  ' if bad==0 else '**FAIL**'} saccades on blink/squint frames = {bad} (must be 0)")

## 10–11. `trials` + `enrich` — the trial table

TRIAL = SPAWN BATCH (one collection → the next). Windows must be **disjoint and contiguous**: unlike JPAS_0231's 3-deep overlapping `df_trials`, these counts may be summed across trials.

In [ ]:
df = pd.read_pickle(SESSION/'df_trials_clean.pkl')
a  = df[df.analyze]
check('trials total', len(df), 79, 2)
check('analyzable', len(a), 77, 2)
for oc, exp in [('single_reward',43), ('banish',17), ('unbanish',17)]:
    check(f'{oc} trials', int((a.outcome==oc).sum()), exp, 4)
gaps = (df.start_frame.values[1:] - df.end_frame.values[:-1])
print(f"  {'PASS  ' if (gaps>=0).all() else '**FAIL**'} windows are disjoint (no negative gap)")
check('total reward units (multiplier sum)', int(a[a.outcome=='single_reward'].multiplier.sum()), 95, 5)
missing = [c for c in ['frac_blink','frac_grooming','whisk_sweep','joy_fine','radius_mean'] if c not in df]
print(f"  {'PASS  ' if not missing else '**FAIL**'} enrich columns present" + (f' — MISSING {missing}' if missing else ''))

## 12. `cluster` — path clustering (and what is NOT a feature)

KMeans on **path geometry only**. Motor / whisker / heading / pupil / **multiplier** are held out so they can be tested as independent outcomes. The assertion below is the anti-circularity guard: if a held-out signal ever appears in `FEATURES`, every downstream finding becomes circular.

In [ ]:
import cluster_paths as cp
held_out = {'joy_fine','whisk_sweep','heading_align','radius_mean','multiplier'}
leak = held_out & set(cp.FEATURES)
print(f"  {'PASS  ' if not leak else '**FAIL**'} no held-out signal is a clustering feature"
      + (f' — LEAKED: {leak}' if leak else ''))
print('  FEATURES =', cp.FEATURES)
df = pd.read_pickle(SESSION/'df_trials_clean.pkl'); a = df[df.analyze]
for nm, exp in [('Direct',30), ('Corner-dwelling',47)]:
    check(f'{nm} trials', int((a.cluster_name==nm).sum()), exp, 8)
# the multiplier must be FLAT across clusters, else it is a confound rather than an independent axis
from scipy.stats import mannwhitneyu
p = mannwhitneyu(a[a.cluster_name=='Direct'].multiplier, a[a.cluster_name=='Corner-dwelling'].multiplier).pvalue
print(f'  note   multiplier by cluster p={p:.3f} '
      f"({'independent axes — good' if p>0.05 else 'DIFFERS — treat as a confound'})")

## 13–16. `labels`, `h1`, `h2`, `valence` — the analyses

The three hypothesis tests. H2 is the positive one; the check below is the one that matters — the reset must FOLLOW the paw movement (a peak at or before 0 would mean whisking predicts steering, the opposite claim).

In [ ]:
import analyze_h2_whisker_reset as h2
import analyze_collection_valence as val
h2.run(str(SESSION))
print()
val.run(str(SESSION))
print('\n  ^ H2: latency must be POSITIVE (whisking FOLLOWS the paw) and the multiplier trend positive.')
print('    valence: the post-hit window is NOT estimable on reward trials (~96% licking) — by design.')

## 17. `report` — the PDF

13 analysis pages + one page per analyzable trial, every page stamped with its number.

In [ ]:
import subprocess
pdf = SESSION/'trial_report.pdf'
print(f"  {'PASS  ' if pdf.exists() else '**FAIL**'} {pdf.name} exists")
if pdf.exists():
    out = subprocess.run(['pdfinfo', str(pdf)], capture_output=True, text=True).stdout
    pages = int([l for l in out.splitlines() if l.startswith('Pages')][0].split()[-1])
    check('pages', pages, 98, 3)

## 17-19. `wcontext`, `perf`, `explain` — whisker context + the PERFORMANCE scorecard

The performance step is the one with a real failure mode: its chance baseline is **visibility-weighted**,
so it depends on `view_scale`. If that is wrong the baseline is wrong and D — the headline learning
metric — is wrong with nothing looking broken. The checks below therefore verify the baseline is
actually the visibility-weighted one (not the 2:1 icon count), that its collection counts agree with
the trial table, and that D is reconstructible from the reported accuracy and chance.

In [ ]:
import analyze_performance as perf
import analyze_whisker_context as wctx
blocks = perf.run(str(SESSION), write=False)
B = blocks['ALL']

# counts must agree with the trial table checked earlier (independent path to the same numbers)
a = pd.read_pickle(SESSION/'df_trials_clean.pkl'); a = a[a.analyze]
check('perf n positive matches trials', B['pos'], int((a.outcome=='single_reward').sum()), 0)
check('perf n negative matches trials', B['neg'], int((a.outcome=='banish').sum()), 0)
check('perf reward units matches trials', B['units'],
      int(a[a.outcome=='single_reward'].multiplier.sum()), 0)

# the baseline must be the VISIBILITY-weighted one, not the 2:1 icon count
ok = abs(B['chance'] - 2/3) > 1e-6
print(f"  {'PASS  ' if ok else '**FAIL**'} chance is visibility-weighted ({B['chance']:.3f}), "
      f"not the 2:1 icon count (0.667)")
check('reward icons visible per frame', round(B['vis_r'], 2), 0.69, 0.10)
check('banish icons visible per frame', round(B['vis_b'], 2), 0.28, 0.10)

# D must be reconstructible from accuracy + chance
D_rebuilt = (B['acc'] - B['chance']) / (1 - B['chance'])
check('D = (acc-chance)/(1-chance)', round(D_rebuilt, 4), round(B['D'], 4), 0.001)
check('discrimination score D', round(B['D'], 3), 0.013, 0.15)
print(f"  note   D={B['D']:+.3f}, p={B['p']:.3f} -> "
      f"{'AT CHANCE (expected for an animal still in training)' if B['p']>0.05 else 'above chance'}")
print(f"  note   against the 2:1 icon-count baseline D would read "
      f"{(B['acc']-2/3)/(1-2/3):+.3f} - the reason the baseline correction matters")

wctx.run(str(SESSION))
for f in ['whisker_context.png', 'performance.png', 'visibility_chance_explainer.png']:
    e = (SESSION/'debug'/f).exists()
    print(f"  {'PASS  ' if e else '**FAIL**'} debug/{f} exists")

## 20. validation CLIPS — the part a number cannot replace

Two renderers, each with two styles. **Style is auto-detected and a mismatched style is refused, not faked** — asking for `stable` iris on an approximate-iris animal, or `pixel` mouth state without a tongue signal, raises rather than drawing a confident overlay on a weak signal.

* `annotate_clip.py` — `approx` (JPAS_0168: dashed amber circle, `~radius`, standing APPROXIMATE banner, no dil/con pills) vs `stable` (JPAS_0231: solid yellow circle, shaded dil/con sparkline, DILATION/CONSTRICTION pills, IRIS BEHIND GLINT + EYE SQUINT status slots).
* `annotate_mouth_clip.py` — `flow` (JPAS_0168: live paw/whisker/mouth-flow bars + a REASSIGNED marker wherever grooming overrode licking) vs `pixel` (JPAS_0231: tongue_out / spout_dark / paw_mouth feature patches + the 4-state axis with TONGUE_OUT).

In [ ]:
import annotate_clip, annotate_mouth_clip
LO, HI = 72800, 73000          # a window with blinks, a squint, licking AND grooming
annotate_clip.main(str(SESSION), LO, HI)              # -> test/eye_clip_approx_*.mp4
annotate_mouth_clip.main(str(SESSION), LO, HI)        # -> test/mouth_clip_flow_*.mp4
print('\n  ⚠️ OPEN BOTH CLIPS. On every REASSIGNED frame the paw must be visibly UP at the face —')
print('     that is the grooming-over-licking priority rule, and it is the one call a statistic cannot make.')
summary()